In [1]:
import pandas as pd
import numpy as np
import os
import sys
import subprocess
import torch
import gc

# --- 1. Install Libraries ---
def install(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", package])

try:
    from sentence_transformers import SentenceTransformer
except ImportError:
    print("Installing sentence-transformers...")
    install("sentence-transformers")
    from sentence_transformers import SentenceTransformer

try:
    import pyarrow as pa
    import pyarrow.parquet as pq
except ImportError:
    print("Installing pyarrow...")
    install("pyarrow")
    import pyarrow as pa
    import pyarrow.parquet as pq

# --- 2. Configuration ---
INPUT_FILE = "/kaggle/input/findsum-sentences/findsum_sentences_output.csv"
OUTPUT_FILE = "/kaggle/working/sentence_embeddings.parquet"
MODEL_NAME = 'all-MiniLM-L6-v2'

def main():
    if not os.path.exists(INPUT_FILE):
        print(f"Error: Input file not found at {INPUT_FILE}")
        return

    print(f"--- Loading Data from {INPUT_FILE} ---")
    df = pd.read_csv(INPUT_FILE)
    print(f"Loaded {len(df):,} sentences.")
    
    # --- 3. Setup Model ---
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"Using device: {device.upper()}")
    
    print(f"Loading model: {MODEL_NAME}...")
    model = SentenceTransformer(MODEL_NAME, device=device)

    # --- 4. Generate Embeddings ---
    print("Generating embeddings...")
    # encode() returns a numpy array by default, which is memory efficient
    embeddings = model.encode(
        df['sentence'].astype(str).tolist(),
        batch_size=128 if device == 'cuda' else 32,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True 
    )
    
    print(f"Embedding shape: {embeddings.shape}")

    # --- 5. Save Results (Memory Optimized) ---
    print("Saving results to Parquet (Optimized Mode)...")
    
    # CRITICAL FIX:
    # Instead of `df['embedding'] = list(embeddings)` which explodes RAM,
    # we build a PyArrow table directly from the numpy array.
    
    # 1. Create Table from existing text data
    table = pa.Table.from_pandas(df)
    
    # 2. Create optimized FixedSizeListArray for vectors
    # This keeps data packed in C-structs rather than Python objects
    dim = embeddings.shape[1]
    tensor_array = pa.FixedSizeListArray.from_arrays(
        embeddings.flatten(), 
        dim
    )
    
    # 3. Append the embeddings column to the table
    table = table.append_column("embedding", tensor_array)
    
    # 4. Write to disk
    pq.write_table(table, OUTPUT_FILE)
    
    print(f"\n--- Success! ---")
    print(f"Embeddings saved to: {OUTPUT_FILE}")
    print(f"File size: {os.path.getsize(OUTPUT_FILE) / (1024*1024):.2f} MB")

if __name__ == "__main__":
    main()

2025-11-29 14:25:39.914616: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764426340.110504      20 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764426340.171472      20 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Generating embeddings...


Batches:   0%|          | 0/36580 [00:00<?, ?it/s]

Embedding shape: (4682238, 384)
Saving results to Parquet (Optimized Mode)...

--- Success! ---
Embeddings saved to: /kaggle/working/sentence_embeddings.parquet
File size: 7293.78 MB
